# Resroduce PyROL results 

This notebook loads the saved POUNDERS trust-region subproblem and calls only the ROL inner trust-region solver. (try setting nf=3, xkin=3, while reading the input file)  

In [15]:
!pip install pyroltrilinos

In [19]:
dir(ROL)

['Algorithm',
 'AlgorithmState',
 'BoundConstraint',
 'Bounds',
 'Constraint',
 'LinMoreAlgorithm',
 'Objective',
 'OptimizationProblem',
 'OptimizationSolver',
 'ParameterList',
 'Problem',
 'Secant',
 'StatusTest',
 'StdVector',
 'TypeBAlgorithm',
 'UpdateType',
 'Vector',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__spec__',
 'lBFGS',
 'load_algorithm',
 'numpy_vector',
 'serialise_algorithm']

In [11]:
from pathlib import Path
import time

import numpy as np
import ROL
from ROL.numpy_vector import NumpyVector

input_path = Path("trsp_debug") / "trsp_pyrol_inputs_nf3_xkin3.npz"
data = np.load(input_path)

G = np.asarray(data["G"], dtype=float).reshape(-1)
H = np.asarray(data["H"], dtype=float)
Lows = np.asarray(data["Lows"], dtype=float).reshape(-1)
Upps = np.asarray(data["Upps"], dtype=float).reshape(-1)
n = int(np.asarray(data["n"]).reshape(()))

print("Loaded:", input_path)
print("nf:", int(np.asarray(data["nf"]).reshape(())))
print("xkin:", int(np.asarray(data["xkin"]).reshape(())))
print("outer POUNDERS delta:", float(np.asarray(data["delta"]).reshape(())))
print("n:", n)
print("G shape:", G.shape)
print("H shape:", H.shape)
print("step bounds:", Lows.min(), Upps.max())

Loaded: trsp_debug/trsp_pyrol_inputs_nf3_xkin3.npz
nf: 3
xkin: 3
outer POUNDERS delta: 0.0125
n: 1616
G shape: (1616,)
H shape: (1616, 1616)
step bounds: -0.0125 0.0125


Below class `TRSPObjective` is taken from the github link shared by Jeff. 

In [12]:
def numpy_vector(values):
    values = np.asarray(values, dtype=float).reshape(-1)
    vec = NumpyVector(values.size)
    vec.data[:] = values
    return vec

class TRSPObjective(ROL.Objective):
    def __init__(self, G, H):
        ROL.Objective.__init__(self)
        self.G = G
        self.H = H

    def value(self, x, tol):
        s = np.asarray(x.data, dtype=float)
        return float(self.G @ s + 0.5 * s @ (self.H @ s))

    def gradient(self, g, x, tol):
        s = np.asarray(x.data, dtype=float)
        g.data[:] = self.G + self.H @ s

    def hessVec(self, hv, v, x, tol):
        hv.data[:] = self.H @ np.asarray(v.data, dtype=float)


objective = TRSPObjective(G, H)
x = numpy_vector(np.zeros(n))
lower = numpy_vector(Lows)
upper = numpy_vector(Upps)

params = ROL.ParameterList(
    {
        "Step": {
            "Type": "Trust Region",
            "Trust Region": {
                "Subproblem Solver": "Truncated CG",
            },
        },
        "General": {"Print Verbosity": 0},
    },
    "Parameters",
)

problem = ROL.OptimizationProblem(objective, x, ROL.Bounds(lower, upper))
solver = ROL.OptimizationSolver(problem, params)

Check line number 84 and 85 in the output. There is a sudden jump. This is what I was talking about. 

In [13]:
t0 = time.perf_counter()
solver.solve()
elapsed = time.perf_counter() - t0

step = np.asarray(x.data, dtype=float)
mdec = objective.value(x, 0.0)

print("\nFinished ROL solve")
print("elapsed seconds:", elapsed)
print("step shape:", step.shape)
print("step norm 2:", np.linalg.norm(step))
print("step norm inf:", np.linalg.norm(step, ord=np.inf))
print("model decrease:", mdec)


Truncated CG Trust-Region Solver
Trust-Region Model: Kelley-Sachs
  iter  value          gnorm          snorm          delta          #fval     #grad     tr_flag   iterCG    flagCG    
  0     0.000000e+00   7.738570e-03                  2.622232e-06   
  1     -1.059085e-08  1.939180e-02   2.622232e-06   6.555579e-06   8         3         0         1         3         
  2     -3.916477e-08  1.321774e-02   6.555579e-06   1.638895e-05   13        5         0         6         3         
  3     -7.113111e-08  1.017431e-02   1.638895e-05   4.097237e-05   18        7         0         8         3         
  4     -9.267663e-08  8.375798e-03   2.320571e-05   1.024309e-04   23        9         0         20        1         
  5     -9.813491e-08  4.389279e-03   8.819552e-06   2.560773e-04   28        11        0         20        1         
  6     -1.003886e-07  3.014511e-03   6.486267e-06   6.401933e-04   33        13        0         20        1         
  7     -1.015386e-07  2.413413